# Cathey — Development & Debug Notebook
Interactive testing of the current pipeline. Run cells top to bottom for full setup, or jump to any section.

**Sections**
1. Setup & Imports
2. LLM — Intent Classification
3. LLM — Clarification Resolution
4. LLM — General QA
5. Rule-Based Fast Path
6. Schema Validation
7. Memory System
8. Full Agent Pipeline (text input)
9. STT (Pi only)
10. TTS (Pi only)

## 1. Setup & Imports

In [1]:
import sys, os, logging, time
logging.getLogger('sentence_transformers').setLevel(logging.ERROR)
logging.getLogger('faster_whisper').setLevel(logging.ERROR)

from sentence_transformers import SentenceTransformer
from llm_parser import LLMParser
from memory import MemoryManager
from agent import CatheyAgent
from rule_based import try_rule_based
from schema import validate_command, execute_command
from config import EMBED_MODEL_NAME

print('Imports OK')

Imports OK


In [2]:
# Load models (slow on first run)
print('Loading LLM...')
llm = LLMParser()

print('Loading embedding model...')
embed = SentenceTransformer(EMBED_MODEL_NAME)
embed.encode('warmup', convert_to_numpy=True)  # pre-warm JIT

memory = MemoryManager(embed_fn=lambda t: embed.encode(t, convert_to_numpy=True).tolist())

# Stub GPIO for non-Pi environments
try:
    from gpio_executor import GPIOExecutor
    gpio = GPIOExecutor()
    print('GPIO ready (Pi)')
except Exception as e:
    gpio = None
    print(f'GPIO unavailable ({e}) — using stub')

# Stub TTS — just prints
def speak(text):
    print(f'[TTS] {text}')

agent = CatheyAgent(llm=llm, memory=memory, speak=speak, gpio=gpio)
print('\nAll components ready.')

Loading LLM...
Loading LLM (Qwen/Qwen2.5-3B-Instruct) on mps [torch.float16] ...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

LLM ready.
Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GPIO unavailable (No module named 'lgpio') — using stub

All components ready.


## 2. LLM — Intent Classification

In [3]:
def classify(text):
    t0 = time.perf_counter()
    result, raw, ms = llm.parse_unified(text, verbose=False)
    print(f'Input : {text}')
    print(f'Output: {result}')
    print(f'Raw   : {raw}')
    print(f'Time  : {ms:.0f} ms\n')
    return result

# Test cases — edit as needed
test_inputs = [
    'Cathey, turn on the light.',
    'Cathey, I feel cold.',
    "Cathey, it's too bright.",
    'Cathey, set the AC to 26 degrees.',
    'Cathey, open the curtain halfway.',
    'Cathey, what time is it?',
    'Hello.',
]

for t in test_inputs:
    classify(t)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Input : Cathey, turn on the light.
Output: {'type': 'direct_command', 'device': 'light', 'action': 'turn_on', 'value': None, 'reply': 'Turning on the light.'}
Raw   : {"type":"direct_command","device":"light","action":"turn_on","value":null,"reply":"Turning on the light."}
Time  : 3012 ms

Input : Cathey, I feel cold.
Output: {'type': 'needs_clarification', 'question': 'Would you like me to close the window or raise the AC temperature?', 'options': ['close_window', 'raise_ac_temperature'], 'reply': 'Would you like me to close the window or raise the AC temperature?'}
Raw   : {"type":"needs_clarification","question":"Would you like me to close the window or raise the AC temperature?","options":["close_window","raise_ac_temperature"]}
Time  : 2963 ms

Input : Cathey, it's too bright.
Output: {'type': 'needs_clarification', 'question': 'Would you like me to lower the brightness or close the curtain?', 'options': ['lower_brightness', 'close_curtain'], 'reply': 'Would you like me to lower t

In [4]:
# Single input test
classify('Cathey, lower the temperature.')

Input : Cathey, lower the temperature.
Output: {'type': 'direct_command', 'device': 'ac', 'action': 'set_temperature', 'value': None, 'reply': 'Lowering the AC temperature.'}
Raw   : {"type":"direct_command","device":"ac","action":"set_temperature","value":null,"reply":"Lowering the AC temperature."}
Time  : 2490 ms



{'type': 'direct_command',
 'device': 'ac',
 'action': 'set_temperature',
 'value': None,
 'reply': 'Lowering the AC temperature.'}

## 3. LLM — Clarification Resolution

In [5]:
def resolve(original, question, options, user_reply):
    result, raw, ms = llm.resolve_followup(
        original_text=original,
        question=question,
        options=options,
        user_reply=user_reply,
        verbose=False
    )
    print(f'Reply : {user_reply}')
    print(f'Output: {result}')
    print(f'Time  : {ms:.0f} ms\n')
    return result

resolve(
    original="Cathey, I feel cold.",
    question="Would you like me to close the window or raise the AC temperature?",
    options=["close_window", "raise_ac_temperature"],
    user_reply="close the window please"
)

resolve(
    original="Cathey, it's too bright.",
    question="Would you like me to lower the brightness or close the curtain?",
    options=["lower_brightness", "close_curtain"],
    user_reply="the second one"
)

Reply : close the window please
Output: {'type': 'direct_command', 'device': 'window', 'action': 'close', 'value': None, 'reply': 'Closing the window.'}
Time  : 2405 ms

Reply : the second one
Output: {'type': 'direct_command', 'device': 'curtain', 'action': 'close', 'value': None, 'reply': 'Closing the curtain.'}
Time  : 2432 ms



{'type': 'direct_command',
 'device': 'curtain',
 'action': 'close',
 'value': None,
 'reply': 'Closing the curtain.'}

## 4. LLM — General QA

In [6]:
def qa(question, context=''):
    answer, ms = llm.answer_qa(question, context=context, verbose=False)
    print(f'Q: {question}')
    print(f'A: {answer}')
    print(f'Time: {ms:.0f} ms\n')
    return answer

qa('What is your name?')
qa('How do I make tea?')

Q: What is your name?
A: My name is Cathey.
Time: 883 ms

Q: How do I make tea?
A: Boil water, add tea leaves or tea bags to a cup, pour hot water over them, let it steep for a few minutes, then add sweetener and milk if you like.
Time: 2703 ms



'Boil water, add tea leaves or tea bags to a cup, pour hot water over them, let it steep for a few minutes, then add sweetener and milk if you like.'

In [7]:
# QA with memory context
context = memory.build_context("what's my name?")
print('Context:\n', context or '(empty)\n')
qa("Cathey, what's my name?", context=context)

Context:
 (empty)

Q: Cathey, what's my name?
A: Your name is not provided here, so I can't determine it. Please tell me your name.
Time: 2057 ms



"Your name is not provided here, so I can't determine it. Please tell me your name."

## 5. Rule-Based Fast Path

In [8]:
rule_tests = [
    'Cathey, turn on the light.',
    'Cathey, open the curtain.',
    'Cathey, open the curtain a little.',
    'Cathey, open the curtain halfway.',
    'Cathey, set the AC to 24 degrees.',
    'Cathey, set brightness to 70.',
    'Cathey, make the light warmer.',
    'Cathey, party time!',
    'Cathey, I feel cold.',
    'Cathey, what time is it?',
]

state = gpio.get_device_state() if gpio else {}
for t in rule_tests:
    result = try_rule_based(t, state)
    tag = '✓ RULE' if result else '→ LLM '
    print(f'{tag}  {t!r:45}  →  {result}')

✓ RULE  'Cathey, turn on the light.'                   →  {'type': 'direct_command', 'device': 'light', 'action': 'turn_on', 'value': None}
✓ RULE  'Cathey, open the curtain.'                    →  {'type': 'direct_command', 'device': 'curtain', 'action': 'open', 'value': None}
✓ RULE  'Cathey, open the curtain a little.'           →  {'type': 'direct_command', 'device': 'curtain', 'action': 'set_position', 'value': 20}
✓ RULE  'Cathey, open the curtain halfway.'            →  {'type': 'direct_command', 'device': 'curtain', 'action': 'set_position', 'value': 50}
✓ RULE  'Cathey, set the AC to 24 degrees.'            →  {'type': 'direct_command', 'device': 'ac', 'action': 'set_temperature', 'value': 24}
✓ RULE  'Cathey, set brightness to 70.'                →  {'type': 'direct_command', 'device': 'light', 'action': 'set_brightness', 'value': 70}
✓ RULE  'Cathey, make the light warmer.'               →  {'type': 'direct_command', 'device': 'light', 'action': 'set_color_temp', 'value': 4}

## 6. Schema Validation

In [9]:
test_commands = [
    {'device': 'light',   'action': 'turn_on',        'value': None},
    {'device': 'light',   'action': 'set_brightness',  'value': 80},
    {'device': 'light',   'action': 'set_color_temp',  'value': 4},
    {'device': 'curtain', 'action': 'set_position',    'value': 50},
    {'device': 'ac',      'action': 'set_temperature', 'value': 24},
    {'device': 'ac',      'action': 'set_temperature', 'value': None},   # invalid
    {'device': 'light',   'action': 'set_brightness',  'value': 150},    # out of range
    {'device': 'none',    'action': 'turn_on',         'value': None},   # unknown device
]

for cmd in test_commands:
    ok, reason = validate_command(cmd)
    status = '✓' if ok else '✗'
    print(f'{status} {str(cmd):60} → {reason}')

✓ {'device': 'light', 'action': 'turn_on', 'value': None}      → ok
✓ {'device': 'light', 'action': 'set_brightness', 'value': 80} → ok
✓ {'device': 'light', 'action': 'set_color_temp', 'value': 4}  → ok
✓ {'device': 'curtain', 'action': 'set_position', 'value': 50} → ok
✓ {'device': 'ac', 'action': 'set_temperature', 'value': 24}   → ok
✗ {'device': 'ac', 'action': 'set_temperature', 'value': None} → value_must_be_int
✗ {'device': 'light', 'action': 'set_brightness', 'value': 150} → value_150_out_of_range_0_100
✗ {'device': 'none', 'action': 'turn_on', 'value': None}       → unknown_device:none


## 7. Memory System

In [10]:
# Memory summary
print('Memory state:', memory.summary())

Memory state: {'working_turns': 0, 'episodic_count': 0, 'prefs': {}, 'skills_count': 0}


In [11]:
# Semantic memory (user prefs)
print('User prefs:', memory.prefs)

User prefs: {}


In [12]:
# Procedural memory (skills)
import json
print('Skills:', json.dumps(memory.skills, indent=2))

Skills: []


In [13]:
# Episodic retrieval test
query = "what's my name?"
if memory.episodes.count() > 0:
    eps = memory.retrieve_episodes(query, n=3)
    for ep in eps:
        print(f"dist={ep['distance']:.3f}  {ep['text'][:60]}  →  {ep['meta']['cathey_reply'][:50]}")
else:
    print('Episodic memory is empty.')

Episodic memory is empty.


## 8. Full Agent Pipeline (text input)

In [14]:
def run(text):
    print(f'>>> {text}')
    result = agent.handle(text, verbose=True)
    print(f'Result: valid={result["valid"]} reason={result["reason"]} exec={result.get("execution")}\n')
    return result

In [15]:
# Direct commands
run('Cathey, turn on the light.')
run('Cathey, open the curtain.')
run('Cathey, set the AC to 28 degrees.')

>>> Cathey, turn on the light.
STT text: Cathey, turn on the light.
[TTS] Sure, turning on the light.
Result: valid=True reason=ok exec=LIGHT -> ON

>>> Cathey, open the curtain.
STT text: Cathey, open the curtain.
[TTS] Sure, opening the curtain.
Result: valid=True reason=ok exec=CURTAIN -> OPEN

>>> Cathey, set the AC to 28 degrees.
STT text: Cathey, set the AC to 28 degrees.
[TTS] Sure, setting AC to 28 degrees.
Result: valid=True reason=ok exec=AC -> TEMPERATURE 28C



{'prefilter_passed': True,
 'semantic': {'type': 'direct_command',
  'device': 'ac',
  'action': 'set_temperature',
  'value': 28,
  'reply': 'Sure, setting AC to 28 degrees.'},
 'valid': True,
 'reason': 'ok',
 'execution': 'AC -> TEMPERATURE 28C',
 'spoken_reply': 'Sure, setting AC to 28 degrees.',
 'llm_latency_ms': 0.0}

In [16]:
# Clarification flow
run("Cathey, I feel cold.")

>>> Cathey, I feel cold.
STT text: Cathey, I feel cold.
Raw output: {"type":"needs_clarification","question":"Would you like me to close the window or raise the AC temperature?","options":["close_window","raise_ac_temperature"]}
Latency: 3389.5 ms
[Clarification options]
  [1] Close the window
  [2] Raise the AC temperature
[TTS] Would you like me to close the window or raise the AC temperature? Here are your options: Option 1: Close the window. Option 2: Raise the AC temperature. Please say the option number or describe what you want.
Result: valid=True reason=clarification_requested exec=PENDING_USER_REPLY



{'prefilter_passed': True,
 'semantic': {'type': 'needs_clarification',
  'question': 'Would you like me to close the window or raise the AC temperature?',
  'options': ['close_window', 'raise_ac_temperature'],
  'reply': 'Would you like me to close the window or raise the AC temperature?'},
 'valid': True,
 'reason': 'clarification_requested',
 'execution': 'PENDING_USER_REPLY',
 'spoken_reply': 'Would you like me to close the window or raise the AC temperature? Here are your options: Option 1: Close the window. Option 2: Raise the AC temperature. Please say the option number or describe what you want.',
 'llm_latency_ms': 3389.53}

In [17]:
# Clarification reply (run after above)
run("close the window please")

>>> close the window please
STT text: close the window please
Raw output: {"type":"direct_command","device":"window","action":"close","value":null,"reply":"Closing the window."}
Latency: 2532.8 ms
[TTS] Closing the window.
Result: valid=True reason=ok exec=WINDOW -> CLOSE



{'prefilter_passed': True,
 'semantic': {'type': 'direct_command',
  'device': 'window',
  'action': 'close',
  'value': None,
  'reply': 'Closing the window.'},
 'valid': True,
 'reason': 'ok',
 'execution': 'WINDOW -> CLOSE',
 'spoken_reply': 'Closing the window.',
 'llm_latency_ms': 2532.816}

In [18]:
# Memory recall
run('Cathey, my name is Alex.')
run("Cathey, what's my name?")

>>> Cathey, my name is Alex.
STT text: Cathey, my name is Alex.
Raw output: {"type":"general_qa","answer":"Nice to meet you, Alex!"}
Latency: 1913.3 ms
[TTS] Nice to meet you, Alex!
Result: valid=True reason=general_qa_answered exec=NO_DEVICE_ACTION

>>> Cathey, what's my name?
STT text: Cathey, what's my name?
Raw output: {"type":"needs_clarification","question":"Would you like me to introduce myself again?","options":["introduce_yourself"]}
Latency: 2695.9 ms
[Clarification options]
  [1] Introduce yourself
[TTS] Would you like me to introduce myself again? Here are your options: Option 1: Introduce yourself. Please say the option number or describe what you want.
Result: valid=True reason=clarification_requested exec=PENDING_USER_REPLY



{'prefilter_passed': True,
 'semantic': {'type': 'needs_clarification',
  'question': 'Would you like me to introduce myself again?',
  'options': ['introduce_yourself'],
  'reply': 'Would you like me to introduce myself again?'},
 'valid': True,
 'reason': 'clarification_requested',
 'execution': 'PENDING_USER_REPLY',
 'spoken_reply': 'Would you like me to introduce myself again? Here are your options: Option 1: Introduce yourself. Please say the option number or describe what you want.',
 'llm_latency_ms': 2695.852}

In [19]:
# General QA
run('Cathey, how do I make tea?')

>>> Cathey, how do I make tea?
STT text: Cathey, how do I make tea?
[State] Cathey re-invoked; abandoning prior clarification.
Raw output: {"type":"general_qa","answer":"You can make tea by boiling water, adding tea leaves or tea bags, and letting it steep for a few minutes. Then add milk and sugar according to your preference."}
Latency: 3927.5 ms
[TTS] You can make tea by boiling water, adding tea leaves or tea bags, and letting it steep for a few minutes. Then add milk and sugar according to your preference.
Result: valid=True reason=general_qa_answered exec=NO_DEVICE_ACTION



{'prefilter_passed': True,
 'semantic': {'type': 'general_qa',
  'answer': 'You can make tea by boiling water, adding tea leaves or tea bags, and letting it steep for a few minutes. Then add milk and sugar according to your preference.'},
 'valid': True,
 'reason': 'general_qa_answered',
 'execution': 'NO_DEVICE_ACTION',
 'spoken_reply': 'You can make tea by boiling water, adding tea leaves or tea bags, and letting it steep for a few minutes. Then add milk and sugar according to your preference.',
 'llm_latency_ms': 3927.509}

## 9. STT (Pi only)

In [20]:
try:
    from audio import STTModel
    import sounddevice as sd
    import numpy as np
    from config import SAMPLE_RATE

    stt = STTModel()
    duration = 4
    print(f'Recording {duration}s — speak now...')
    audio = sd.rec(int(duration * SAMPLE_RATE), samplerate=SAMPLE_RATE,
                   channels=1, dtype='float32')
    sd.wait()
    text = stt.transcribe(audio)
    print('Transcription:', text)
except Exception as e:
    print(f'STT unavailable: {e}')

Loading STT (tiny.en) ...
STT ready.
Recording 4s — speak now...
Transcription: No, I feel like...


## 10. TTS (Pi only)

In [21]:
try:
    from audio import TTSEngine
    tts_engine = TTSEngine()
    tts_engine.speak('Hello, I am Cathey, your smart home assistant.')
except Exception as e:
    print(f'TTS unavailable: {e}')

TTS unavailable: No module named 'piper'
